# Ch12 — 相關分析與迴歸入門

> **預估時間：2 小時**  
> **前置章節：Ch07 描述統計（Descriptive Statistics）**

---

## 學習目標

完成本章後，你將能夠：

1. 透過**散佈圖（Scatter Plot）**直觀判斷兩變數之間的關係型態
2. 計算並解讀 **Pearson 相關係數（r）** 與 **Spearman 等級相關係數（$\rho$）**
3. 對相關係數進行**顯著性檢定（Significance Test）**，並理解 p-value 的意義
4. 建立**簡單線性迴歸模型（Simple Linear Regression）**，解讀 OLS 結果
5. 理解 **$R^2$ 決定係數（Coefficient of Determination）** 的含義
6. 區分「相關（Correlation）」與「因果（Causation）」

## 環境設定

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# 繪圖設定
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# 隨機種子（確保可重現）
np.random.seed(42)

print('環境載入完成')

## 資料載入

In [ ]:
# Titanic 資料集
titanic = pd.read_csv('datasets/titanic_train.csv')
print(f'Titanic shape: {titanic.shape}')
titanic.head()

In [ ]:
# 電商訂單資料集
ecom = pd.read_csv('datasets/ecommerce_orders.csv')
print(f'Ecommerce shape: {ecom.shape}')
ecom.head()

In [ ]:
# 快速瀏覽 Titanic 數值欄位
titanic.describe()

In [ ]:
# 電商資料結構
ecom.info()

In [ ]:
# 電商數值欄位概覽
ecom.describe()

---

## Part A — 散佈圖與視覺判斷（Scatter Plot & Visual Inspection）

在計算任何數字之前，**先畫圖**永遠是好習慣。散佈圖能幫助我們快速辨識：

| 型態 | 特徵 |
|------|------|
| **正相關（Positive Correlation）** | 一個變數增加，另一個也傾向增加 |
| **負相關（Negative Correlation）** | 一個變數增加，另一個傾向減少 |
| **無相關（No Correlation）** | 兩變數之間看不出明顯趨勢 |
| **非線性（Non-linear）** | 有關係，但不是直線關係 |

In [ ]:
# 用模擬資料展示四種相關型態
np.random.seed(0)
n_demo = 200

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 正相關
x = np.random.normal(0, 1, n_demo)
y_pos = 0.8 * x + np.random.normal(0, 0.4, n_demo)
axes[0].scatter(x, y_pos, alpha=0.5, s=15)
axes[0].set_title('正相關 (Positive)')

# 負相關
y_neg = -0.8 * x + np.random.normal(0, 0.4, n_demo)
axes[1].scatter(x, y_neg, alpha=0.5, s=15, color='orange')
axes[1].set_title('負相關 (Negative)')

# 無相關
y_none = np.random.normal(0, 1, n_demo)
axes[2].scatter(x, y_none, alpha=0.5, s=15, color='green')
axes[2].set_title('無相關 (None)')

# 非線性
y_nonlin = x**2 + np.random.normal(0, 0.3, n_demo)
axes[3].scatter(x, y_nonlin, alpha=0.5, s=15, color='red')
axes[3].set_title('非線性 (Non-linear)')

plt.tight_layout()
plt.show()

In [ ]:
# Titanic: Age vs Fare 散佈圖
titanic_clean = titanic.dropna(subset=['Age', 'Fare'])

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=titanic_clean,
    x='Age', y='Fare',
    hue='Pclass',
    palette='viridis',
    alpha=0.6
)
plt.title('Titanic: Age vs Fare（依艙等著色）')
plt.xlabel('年齡 (Age)')
plt.ylabel('票價 (Fare)')
plt.legend(title='Pclass')
plt.show()

In [ ]:
# Titanic 數值欄位的 pairplot（一次看所有兩兩關係）
selected_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'Survived']
sns.pairplot(titanic_clean[selected_cols].dropna(), hue='Survived', 
             diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15},
             height=2)
plt.suptitle('Titanic 數值變數 Pairplot', y=1.02)
plt.show()

> **口訣：「先畫圖，再算數字」**  
> 光看相關係數可能會被誤導（例如 Anscombe's Quartet），散佈圖能揭示資料的真實樣貌。

---

## Part B — Pearson 相關係數（r）

**Pearson 相關係數（Pearson Correlation Coefficient）** 衡量兩個連續變數之間的**線性**相關強度：

$$
r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \sum_{i=1}^{n}(y_i - \bar{y})^2}}
$$

### 解讀準則

| |r| 範圍 | 強度 |
|-----------|------|
| 0.00 ~ 0.19 | 極弱或無相關 |
| 0.20 ~ 0.39 | 弱相關 |
| 0.40 ~ 0.59 | 中等相關 |
| 0.60 ~ 0.79 | 強相關 |
| 0.80 ~ 1.00 | 極強相關 |

> $r \in [-1, 1]$：正值表示正相關，負值表示負相關，0 表示無線性相關。

In [ ]:
# 計算 Titanic Age vs Fare 的 Pearson 相關係數
r, p_value = stats.pearsonr(titanic_clean['Age'], titanic_clean['Fare'])

print(f'Pearson r = {r:.4f}')
print(f'p-value   = {p_value:.4f}')

if p_value < 0.05:
    print('=> 在 alpha=0.05 下，Age 與 Fare 存在顯著線性相關')
else:
    print('=> 在 alpha=0.05 下，無法拒絕「Age 與 Fare 無線性相關」的虛無假設')

In [ ]:
# Titanic 數值欄位的相關矩陣
numeric_cols = titanic.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = titanic[numeric_cols].corr(method='pearson')
corr_matrix.round(3)

In [ ]:
# 相關矩陣熱力圖（Heatmap）
plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # 遮蔽上三角
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title('Titanic 數值變數相關矩陣')
plt.tight_layout()
plt.show()

In [ ]:
# 手動計算 Pearson r（理解公式）
x_vals = titanic_clean['Age'].values
y_vals = titanic_clean['Fare'].values

x_mean = x_vals.mean()
y_mean = y_vals.mean()

numerator = np.sum((x_vals - x_mean) * (y_vals - y_mean))
denominator = np.sqrt(np.sum((x_vals - x_mean)**2) * np.sum((y_vals - y_mean)**2))

r_manual = numerator / denominator
print(f'手動計算 Pearson r = {r_manual:.4f}')
print(f'scipy 計算 Pearson r = {r:.4f}')
print(f'兩者一致：{np.isclose(r_manual, r)}')

---

## Part C — Spearman 等級相關（$\rho$）

**Spearman 等級相關係數（Spearman Rank Correlation）** 衡量兩變數之間的**單調（Monotonic）**關係——不一定要是線性的。

### 何時用 Spearman？

- 資料為**順序尺度（Ordinal Scale）**（例如教育程度、滿意度評分）
- 資料**不符合常態分佈**
- 存在**極端值（Outliers）**
- 關係是單調但非線性的

計算方式：先將原始資料轉為**等級（Rank）**，再對等級計算 Pearson r。

In [ ]:
# Spearman 等級相關：Age vs Fare
rho, p_spearman = stats.spearmanr(titanic_clean['Age'], titanic_clean['Fare'])

print(f'Spearman rho = {rho:.4f}')
print(f'p-value      = {p_spearman:.4f}')

### 補充：Kendall Tau ($\tau$)

除了 Pearson 和 Spearman，還有 **Kendall Tau** 相關係數。它基於「一致對（Concordant Pairs）」和「不一致對（Discordant Pairs）」的比較：

$$\tau = \frac{(\text{concordant pairs}) - (\text{discordant pairs})}{\binom{n}{2}}$$

- 適用場景與 Spearman 類似，但在**小樣本**時統計性質更好
- 使用 `stats.kendalltau(x, y)` 計算

In [ ]:
# Kendall Tau 計算
tau, p_kendall = stats.kendalltau(titanic_clean['Age'], titanic_clean['Fare'])
print(f'Kendall tau = {tau:.4f}')
print(f'p-value     = {p_kendall:.4f}')

In [ ]:
# Pearson vs Spearman 比較
comparison = pd.DataFrame({
    '指標': ['Pearson r', 'Spearman rho'],
    '數值': [r, rho],
    'p-value': [p_value, p_spearman],
    '衡量對象': ['線性相關', '單調相關'],
    '假設': ['連續、常態分佈', '順序尺度即可'],
    '對極端值敏感度': ['高', '低']
})
comparison

In [ ]:
# 示範：非線性但單調的關係
np.random.seed(7)
x_mono = np.linspace(1, 10, 100)
y_mono = np.log(x_mono) * 3 + np.random.normal(0, 0.3, 100)  # 對數關係

r_pearson, _ = stats.pearsonr(x_mono, y_mono)
r_spearman, _ = stats.spearmanr(x_mono, y_mono)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_mono, y_mono, alpha=0.6, s=20)
ax.set_title(f'對數關係：Pearson r={r_pearson:.3f}, Spearman rho={r_spearman:.3f}')
ax.set_xlabel('X')
ax.set_ylabel('Y = log(X) + noise')
plt.tight_layout()
plt.show()

print(f'Pearson r  = {r_pearson:.4f}  （線性衡量，會低估非線性單調關係）')
print(f'Spearman rho = {r_spearman:.4f}  （單調衡量，更能捕捉此關係）')

---

## Part D — 相關係數的顯著性檢定

### 假設設定

- $H_0$：$\rho = 0$（母體中兩變數無相關）
- $H_1$：$\rho \neq 0$（母體中兩變數存在相關）

### p-value 解讀

- **p < 0.05**：在 5% 顯著水準下，有足夠證據拒絕 $H_0$，認為存在顯著相關
- **p >= 0.05**：無法拒絕 $H_0$，不能斷言存在相關

### 注意：大樣本的陷阱

當樣本量很大時，即使**極微弱**的相關也可能產生很小的 p-value（統計顯著），但在實務上可能**毫無意義**。  
因此，除了看 p-value，也要看 **r 值的大小**。

In [ ]:
# 示範：大樣本 + 微弱相關 → 仍然「顯著」
np.random.seed(99)
n_large = 10000
x_large = np.random.normal(0, 1, n_large)
y_large = 0.03 * x_large + np.random.normal(0, 1, n_large)  # r 約 0.03

r_large, p_large = stats.pearsonr(x_large, y_large)
print(f'樣本量 n = {n_large}')
print(f'Pearson r = {r_large:.4f}  （極弱相關）')
print(f'p-value   = {p_large:.6f}')
print()
if p_large < 0.05:
    print('統計上「顯著」，但 r 值極小 → 實務意義不大！')
    print('這就是為什麼不能只看 p-value。')

In [ ]:
# 效果量（Effect Size）的重要性
# 比較不同樣本量下，相同 r 值的 p-value 變化
sample_sizes = [20, 50, 100, 500, 1000, 5000]
fixed_r = 0.15  # 弱相關

print(f'固定 r = {fixed_r}（弱相關）\n')
print(f'{"n":>6}  {"t-stat":>8}  {"p-value":>10}  {"顯著?":>6}')
print('-' * 40)
for n_s in sample_sizes:
    t_val = fixed_r * np.sqrt((n_s - 2) / (1 - fixed_r**2))
    p_val = 2 * (1 - stats.t.cdf(abs(t_val), df=n_s-2))
    sig = 'Yes' if p_val < 0.05 else 'No'
    print(f'{n_s:>6}  {t_val:>8.3f}  {p_val:>10.6f}  {sig:>6}')

### 知識補給站：相關不等於因果（Correlation does not imply Causation）

即使兩變數高度相關，也**不能**直接推論其中一個「導致」另一個。可能的原因：

1. **干擾變數（Confounding Variable）**：第三個變數同時影響 X 和 Y  
   - 例：冰淇淋銷量與溺水人數正相關 → 干擾變數是「夏季高溫」

2. **反向因果（Reverse Causation）**：可能是 Y 影響 X，而非 X 影響 Y

3. **巧合（Spurious Correlation）**：純粹統計上的巧合

要建立因果關係，需要**隨機對照實驗（Randomized Controlled Trial, RCT）**或嚴謹的因果推論方法。

---

## Part E — 簡單線性迴歸（Simple Linear Regression）

### 模型

$$
y = \beta_0 + \beta_1 x + \varepsilon
$$

- $\beta_0$：截距（Intercept）
- $\beta_1$：斜率（Slope） — x 每增加 1 單位，y 平均變化量
- $\varepsilon$：誤差項（Error Term）

### 最小平方法（OLS, Ordinary Least Squares）

找到一條直線，使得所有觀察值到迴歸線的**殘差平方和**最小：

$$
\min_{\beta_0, \beta_1} \sum_{i=1}^{n} (y_i - \beta_0 - \beta_1 x_i)^2
$$

In [ ]:
# 電商資料：unit_price → amount 的簡單線性迴歸
X = ecom['unit_price']
y = ecom['amount']

# 先看散佈圖
plt.figure(figsize=(8, 5))
sns.scatterplot(x=X, y=y, alpha=0.5, s=20)
plt.xlabel('單價 (unit_price)')
plt.ylabel('金額 (amount)')
plt.title('電商訂單：單價 vs 金額')
plt.show()

In [ ]:
# 建立 OLS 迴歸模型
X_const = sm.add_constant(X)  # 加入截距項
model = sm.OLS(y, X_const).fit()

# 輸出完整摘要
print(model.summary())

### 如何解讀 OLS Summary

| 項目 | 意義 |
|------|------|
| **coef (const)** | $\hat{\beta}_0$ 截距：當 unit_price = 0 時，amount 的預測值 |
| **coef (unit_price)** | $\hat{\beta}_1$ 斜率：unit_price 每增加 1 元，amount 平均變化量 |
| **P>\|t\|** | 該係數是否顯著不等於 0 |
| **R-squared** | 模型解釋了多少百分比的 y 變異 |
| **F-statistic** | 整體模型是否顯著 |

In [ ]:
# 提取關鍵指標
beta_0 = model.params['const']
beta_1 = model.params['unit_price']
r_squared = model.rsquared
p_beta1 = model.pvalues['unit_price']

print(f'截距 (beta_0)     = {beta_0:.4f}')
print(f'斜率 (beta_1)     = {beta_1:.4f}')
print(f'R-squared         = {r_squared:.4f}')
print(f'beta_1 的 p-value = {p_beta1:.2e}')
print()
print(f'解讀：unit_price 每增加 1 元，amount 平均增加 {beta_1:.2f} 元')

In [ ]:
# 迴歸線視覺化
plt.figure(figsize=(8, 5))
sns.scatterplot(x=X, y=y, alpha=0.4, s=20, label='觀察值')

# 繪製迴歸線
x_line = np.linspace(X.min(), X.max(), 100)
y_line = beta_0 + beta_1 * x_line
plt.plot(x_line, y_line, color='red', linewidth=2, label=f'OLS: y = {beta_0:.1f} + {beta_1:.2f}x')

plt.xlabel('單價 (unit_price)')
plt.ylabel('金額 (amount)')
plt.title('簡單線性迴歸：unit_price → amount')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 使用模型進行預測
new_prices = pd.DataFrame({'unit_price': [25.0, 50.0, 100.0, 150.0]})
new_prices_const = sm.add_constant(new_prices)
predictions = model.predict(new_prices_const)

result = new_prices.copy()
result['predicted_amount'] = predictions.round(2)
print('新資料的預測結果：')
result

### OLS 迴歸的四大假設

使用 OLS 迴歸時，需要檢查以下假設是否合理：

1. **線性（Linearity）**：X 與 Y 之間是線性關係
2. **獨立性（Independence）**：殘差之間互相獨立
3. **同質變異（Homoscedasticity）**：殘差的變異在各 X 值下大致相同
4. **常態性（Normality）**：殘差近似常態分佈

若假設嚴重違反，模型的推論（p-value、信賴區間）可能不可靠。

---

## Part F — $R^2$ 決定係數（Coefficient of Determination）

### 定義

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}
$$

- $R^2 = 1$：模型完美解釋所有變異
- $R^2 = 0$：模型沒有解釋任何變異（和直接用平均值預測一樣）

### Adjusted $R^2$（校正決定係數）

當加入更多自變數時，$R^2$ 一定不會下降（即使新變數無用）。Adjusted $R^2$ 對自變數數量做懲罰：

$$
\bar{R}^2 = 1 - \frac{(1 - R^2)(n - 1)}{n - k - 1}
$$

其中 $n$ 為樣本數，$k$ 為自變數個數。

In [ ]:
# 從模型物件中提取 R-squared 與 Adjusted R-squared
print(f'R-squared          = {model.rsquared:.4f}')
print(f'Adjusted R-squared = {model.rsquared_adj:.4f}')
print()
print(f'模型解釋了約 {model.rsquared * 100:.1f}% 的 amount 變異。')

In [ ]:
# 殘差分佈：檢查模型假設
residuals = model.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 殘差 vs 預測值
axes[0].scatter(model.fittedvalues, residuals, alpha=0.4, s=15)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('預測值 (Fitted Values)')
axes[0].set_ylabel('殘差 (Residuals)')
axes[0].set_title('殘差 vs 預測值')

# 殘差的 Q-Q 圖
from scipy.stats import probplot
probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('殘差 Q-Q 圖')

plt.tight_layout()
plt.show()

---

## Part G — 迴歸係數的假設檢定

### 斜率 $\beta_1$ 的檢定

- $H_0$：$\beta_1 = 0$（X 對 Y 沒有線性影響）
- $H_1$：$\beta_1 \neq 0$

### t 統計量

$$
t = \frac{\hat{\beta}_1}{SE(\hat{\beta}_1)}
$$

- $SE(\hat{\beta}_1)$：斜率估計的標準誤
- 自由度 $df = n - 2$（簡單迴歸）

### 和相關係數檢定的關係

在簡單線性迴歸中，**斜率 $\beta_1$ 的 t 檢定**和 **Pearson r 的顯著性檢定**會得到**完全相同的 p-value**。

因為：
$$
t = r \sqrt{\frac{n - 2}{1 - r^2}}
$$

In [ ]:
# 驗證：迴歸 t 檢定 vs Pearson r 檢定
# 迴歸模型的 t 統計量和 p-value
t_regression = model.tvalues['unit_price']
p_regression = model.pvalues['unit_price']

# Pearson 相關的結果
r_corr, p_corr = stats.pearsonr(ecom['unit_price'], ecom['amount'])

# 用公式計算的 t
n_obs = len(ecom)
t_from_r = r_corr * np.sqrt((n_obs - 2) / (1 - r_corr**2))

print('=== 比較三者 ===')
print(f'迴歸 t 統計量     = {t_regression:.4f}')
print(f'由 r 計算的 t     = {t_from_r:.4f}')
print(f'Pearson r         = {r_corr:.4f}')
print()
print(f'迴歸 p-value      = {p_regression:.2e}')
print(f'Pearson p-value   = {p_corr:.2e}')
print()
print('兩者的 p-value 完全相同！在簡單線性迴歸中，斜率檢定等價於相關係數檢定。')

In [ ]:
# 迴歸係數的信賴區間
conf_int = model.conf_int(alpha=0.05)
conf_int.columns = ['下界 (2.5%)', '上界 (97.5%)']
conf_int

---

## 重點整理

### Pearson vs Spearman 速查表

| 特性 | Pearson r | Spearman $\rho$ |
|------|-----------|------------------|
| **衡量對象** | 線性相關 | 單調相關 |
| **資料要求** | 連續、近似常態 | 順序尺度即可 |
| **極端值影響** | 敏感 | 較穩健 |
| **計算方式** | 原始資料 | 等級化後計算 |
| **適用情境** | 正態連續資料 | 非常態、有極端值、順序資料 |

### 本章核心概念

1. **先畫圖，再算數字** — 散佈圖是相關分析的起點
2. **相關係數**只衡量兩變數的關聯強度，不代表因果
3. **p-value 顯著 $\neq$ 實務重要** — 大樣本下微弱相關也會「顯著」
4. **OLS 迴歸**是建模預測的基礎，但要檢查殘差假設
5. **$R^2$** 告訴你模型解釋了多少變異，Adjusted $R^2$ 更適合比較不同模型

---

## 練習題

### 基礎題

1. 使用 Titanic 資料，計算 `SibSp` 與 `Parch` 之間的 Pearson 相關係數，並判斷是否顯著。

2. 繪製 ecommerce 資料中 `unit_price` 與 `rating` 的散佈圖，目測是否有相關？

### 進階題

3. 分別計算 Titanic 中 `Age` vs `Fare` 的 Pearson r 和 Spearman $\rho$，哪個較大？為什麼？

4. 使用 ecommerce 資料，以 `rating` 為自變數、`amount` 為依變數建立 OLS 迴歸模型。$R^2$ 有多大？rating 對 amount 有顯著影響嗎？

### 挑戰題

5. 生成一組 $y = x^2$ 的資料（加入適量雜訊），分別計算 Pearson 和 Spearman 相關係數。解釋為什麼兩者差異很大，並說明哪個更能反映資料的真實關係。

6. 在電商資料中，`amount = unit_price * quantity + noise`。建立以 `unit_price` 和 `quantity` 為自變數的**多元迴歸**模型（提示：`sm.OLS(y, sm.add_constant(X_multi))`），比較其 Adjusted $R^2$ 與簡單迴歸的差異。

In [ ]:
# 練習空間
# 請在此撰寫你的解答


In [ ]:
# 練習空間（進階題）
# 請在此撰寫你的解答


In [ ]:
# 練習空間（挑戰題）
# 請在此撰寫你的解答


---

| [< Ch11 卡方檢定](11_卡方檢定.ipynb) | [Ch13 統計檢定決策樹與 AB Testing 實戰 >](13_統計檢定決策樹與AB_testing實戰.ipynb) |